# Transformer vs Mamba: full study runner

Runs the whole study on a **free** Kaggle GPU (main) or a free Colab GPU (backup). You choose what
one session does in the **first code cell** (`STAGES`), then run everything top to bottom.

**On Kaggle (recommended)**
1. Settings panel: *Accelerator* = **GPU T4 x2** (both GPUs are used: two runs train at once), *Internet* = **On**.
   The `data` stage needs no GPU: run it with *Accelerator* = **None** to save GPU quota.
2. Edit the first code cell, then **Save Version → Save & Run All**. It keeps running with the browser closed.
3. Next session: **Add Input** → the *data* version's output **and** the latest version's output.
   Finished runs, paused runs (they resume from `last.pt`) and results are picked up automatically.
4. Training pauses cleanly before Kaggle's 12-hour limit (`SESSION_HOURS`), so nothing is lost.

**Splitting work across the 4 team accounts.** Each account runs different runs by setting `ONLY_RUNS`
(for example `{'S_mamba_s1', 'S_mamba_lti_s1'}`), then everyone adds the others' outputs as inputs.

| Stage | What it produces | Time on Kaggle T4 x2 |
| --- | --- | --- |
| `data` | 2B FineWeb-Edu training tokens + 50 PG-19 books, 16K BPE | 1.1 h measured, once (CPU session) |
| `smoke` | 20-step speed/memory check and time estimates for every stage | 0.3 h measured |
| `sweep` | best learning rate per architecture at S (47M): Transformer x 5 and Mamba x 4 LRs, 30M tokens each | about 2.5-3 h |
| `S` | Transformer, Mamba, Mamba-LTI x seeds, 340M tokens each (~47M) | about 9.5 h per run, two runs at once |
| `synthetic` | MQAR and selective-copying grids | ~1-3 h |
| `eval` | test perplexity, recall split, PG-19, efficiency, delta analysis | ~1-2 h |
| `M` | Transformer and Mamba, 1 seed (~98M) | not in the current plan: ~100 h per run on a T4 |
| `figures` | `results/figures/*`, `table_main.md` and `results.zip` | minutes |

Study order: `data` → `smoke` → `sweep` → `S` (split over accounts with `ONLY_RUNS`) → `synthetic` + `eval` + `figures`.
Speeds measured by the smoke test (2026-11-08): ~10,400 tokens/s per T4 for both models with the
`mamba-ssm` kernel, 380 tokens/s for Mamba with the pure-PyTorch chunked scan.

In [ ]:
# ===== Settings for this session: edit, then Save Version -> Save & Run All =====
# Stages: data, smoke, sweep, S, synthetic, eval, M, figures (see the table above)
STAGES = {'data'}
SESSION_HOURS = 11.0   # Kaggle stops a session at 12 h; training pauses cleanly (saves last.pt) before this
USE_MAMBA_SSM = True   # try the official fused Mamba kernel; without it our chunked PyTorch scan is used
ONLY_RUNS = None       # to split work across accounts, e.g. {'S_mamba_s1', 'S_mamba_lti_s1'}; None = all runs
REPO_URL = 'https://github.com/Gagan-1718/afml-transformer-vs-state-space-models.git'
BRANCH = 'upgrade/full-study'

import time
T0 = time.time()
DEADLINE = T0 + SESSION_HOURS * 3600

In [ ]:
import os, sys, subprocess, json, glob, shutil, csv, shlex
from pathlib import Path

ON_KAGGLE = os.path.exists('/kaggle')
if ON_KAGGLE:
    BASE = Path('/kaggle/working/afml')   # everything here is saved with the version
    REPO = Path('/tmp/afml-repo')         # code lives outside the output
else:  # Colab: keep everything on Google Drive so it survives disconnects
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/afml')
    REPO = Path('/content/afml-repo')
DATA = BASE / 'data_fineweb_16k'
RUNS = BASE / 'runs'
RESULTS = BASE / 'results'
WHEELS = BASE / 'wheels'
for p in (BASE, RUNS, RESULTS):
    p.mkdir(parents=True, exist_ok=True)
PY = sys.executable
PAUSED = 75  # exit code of afml.train when it paused at the session deadline
DIVERGED = 3  # exit code of afml.train when the loss became NaN/inf (it writes diverged.json)


def clock():
    return time.strftime('%H:%M')


def hours_left():
    return (DEADLINE - time.time()) / 3600


def sh(cmd, cwd=None, ok=(0,)):
    # Run a shell command, stream its output into the notebook, and stop on failure.
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd or REPO, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    code = p.wait()
    if code not in ok:
        raise RuntimeError(f'command failed (exit {code}): {cmd}')
    return code


if not REPO.exists():
    sh(f'git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO}', cwd='/')
else:
    sh(f'git pull --ff-only origin {BRANCH}')
os.chdir(REPO)
sys.path[:0] = [str(REPO), str(REPO / 'scripts')]
sh('git log -1 --format="code version: %h %s"')
sh('pip install -q -r requirements.txt')

try:
    GPU_NAMES = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                               capture_output=True, text=True, check=True).stdout.strip().splitlines()
except (OSError, subprocess.CalledProcessError):
    GPU_NAMES = []
GPUS = list(range(len(GPU_NAMES))) or [None]
print('GPUs:', GPU_NAMES or 'none (CPU session: fine for the data stage only)')

if ON_KAGGLE:  # optional: a Hugging Face token avoids anonymous rate limits (Add-ons -> Secrets -> HF_TOKEN)
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
        print('using HF_TOKEN from Kaggle secrets')
    except Exception:
        pass

**Reuse earlier sessions (Kaggle).** Kaggle wipes `/kaggle/working` between sessions. This cell finds the
dataset and copies runs and results from every output added with *Add Input*. If a run appears in
several inputs, the most advanced copy wins. Finished runs keep only `best.pt` (all the evaluations need),
and finished sweep/smoke runs keep no weights, which keeps the output far below Kaggle's 20 GB limit.

In [ ]:
def is_disposable(rel):
    # Sweep and smoke runs are only needed for their summaries once finished.
    return Path(rel).parts[0] == 'sweep' or Path(rel).name.startswith('smoke_')


def finished(run):
    # A finished run has summary.json; a sweep run whose loss blew up has diverged.json instead.
    return (Path(run) / 'summary.json').exists() or (Path(run) / 'diverged.json').exists()


def progress(run):
    run = Path(run)
    if finished(run):
        return (1, 0)
    m = run / 'metrics.csv'
    return (0, len(m.read_text().splitlines()) if m.exists() else 0)


def tidy(run, keep):
    for f in Path(run).glob('*.pt'):
        if f.name not in keep:
            f.unlink()


if ON_KAGGLE:
    found = sorted(glob.glob('/kaggle/input/**/data_fineweb_16k/meta.json', recursive=True))
    if found and not (DATA / 'meta.json').exists():
        DATA = Path(found[0]).parent
    best = {}
    for root in glob.glob('/kaggle/input/**/afml/runs', recursive=True):
        for cfg in Path(root).rglob('config.json'):
            rel = cfg.parent.relative_to(root)
            if rel not in best or progress(cfg.parent) > progress(best[rel]):
                best[rel] = cfg.parent
    restored = 0
    for rel, src in sorted(best.items()):
        dst = RUNS / rel
        if dst.exists() and progress(dst) >= progress(src):
            continue
        if dst.exists():
            shutil.rmtree(dst)
        done = finished(src)
        drop = ({'last.pt'} if done else set()) | ({'best.pt'} if done and is_disposable(rel) else set())
        shutil.copytree(src, dst, ignore=lambda d, names, drop=drop: [n for n in names if n in drop])
        restored += 1
    for src in sorted(glob.glob('/kaggle/input/**/afml/results', recursive=True)):
        shutil.copytree(src, RESULTS, dirs_exist_ok=True)
    for src in sorted(glob.glob('/kaggle/input/**/afml/wheels', recursive=True)):
        shutil.copytree(src, WHEELS, dirs_exist_ok=True)
    print('data:', DATA, '| runs restored:', restored)

**Runner.** Training jobs run in parallel, one per GPU (`T4 x2` → two runs at once). Each job's full log
goes to `train.log` inside its run folder; the notebook prints starts, finishes and a progress line
every 15 minutes. Jobs get `--stop-at` = the session deadline, so they save `last.pt` and pause in time.

In [ ]:
def tail(path, n=1):
    try:
        lines = Path(path).read_text(errors='replace').splitlines()
    except OSError:
        return ''
    return '\n'.join(lines[-n:])


def train_job(name, config, out, sets=(), keep=('best.pt',)):
    args = [PY, '-m', 'afml.train', '--config', str(config), '--data', str(DATA), '--out', str(out)]
    for s in sets:
        args += ['--set', s]
    return {'name': name, 'cmd': args, 'out': Path(out), 'log': Path(out) / 'train.log',
            'pausable': True, 'filter': True, 'keep': keep}


def run_jobs(jobs, min_hours=0.3):
    # Run jobs in parallel, one per GPU, until all finish or the session runs out of time.
    # Returns True when every job finished. Raises at the end if any job failed.
    pending, skipped = [], []
    for j in jobs:
        if j['filter'] and ONLY_RUNS is not None and j['name'] not in ONLY_RUNS:
            skipped.append(j['name'])
            continue
        if j.get('out') is not None and (j['out'] / 'summary.json').exists():
            print('done earlier:', j['name'])
        elif j.get('diverged_ok') and (j['out'] / 'diverged.json').exists():
            print('diverged earlier (learning rate too high):', j['name'])
        else:
            pending.append(j)
    if skipped:
        print('left to other accounts (ONLY_RUNS):', skipped)
    running, failed, paused, beat = {}, [], [], time.time()
    while pending or running:
        for g in GPUS:
            if g in running or not pending or hours_left() < min_hours:
                continue
            j = pending.pop(0)
            cmd = [str(c) for c in j['cmd']] + (['--stop-at', f'{DEADLINE:.0f}'] if j['pausable'] else [])
            env = dict(os.environ, PYTHONUNBUFFERED='1')
            if g is not None:
                env['CUDA_VISIBLE_DEVICES'] = str(g)
            j['log'].parent.mkdir(parents=True, exist_ok=True)
            fh = open(j['log'], 'a')
            fh.write(f'\n[{time.ctime()}] $ {shlex.join(cmd)}\n')
            fh.flush()
            proc = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=fh, stderr=subprocess.STDOUT)
            running[g] = (j, proc, fh)
            print(f'[{clock()}] start {j["name"]} on GPU {g} ({hours_left():.1f} h left)', flush=True)
        if not running:
            break
        time.sleep(20)
        for g, (j, proc, fh) in list(running.items()):
            code = proc.poll()
            if code is None:
                continue
            fh.close()
            del running[g]
            if code == 0:
                print(f'[{clock()}] finished {j["name"]}: {tail(j["log"])}', flush=True)
                if j.get('out') is not None:
                    tidy(j['out'], j['keep'])
            elif code == PAUSED:
                paused.append(j['name'])
                print(f'[{clock()}] paused {j["name"]} at the session limit; it resumes next session', flush=True)
            elif code == DIVERGED and j.get('diverged_ok'):
                print(f'[{clock()}] diverged {j["name"]}: {tail(j["log"])} '
                      '(normal in a sweep: this learning rate is too high; not an error)', flush=True)
                tidy(j['out'], ())
            else:
                failed.append(j['name'])
                print(f'[{clock()}] FAILED {j["name"]} (exit {code}). Last lines of {j["log"]}:\n'
                      f'{tail(j["log"], 30)}', flush=True)
        if running and time.time() - beat > 900:
            beat = time.time()
            for g, (j, _, _) in running.items():
                print(f'[{clock()}] {j["name"]} (GPU {g}): {tail(j["log"])}', flush=True)
    if pending:
        print('not started (not enough session time left):', [j['name'] for j in pending])
    if failed:
        raise RuntimeError(f'failed jobs: {failed}. Their logs are printed above; fix the cause, then rerun.')
    return not (pending or paused)

**The official Mamba CUDA kernel (`mamba-ssm`): needed in practice.** On a T4, our pure-PyTorch chunked
scan is memory-bound and far slower than the Transformer, so the study's Mamba runs need the fused kernel.
Install order, fastest first:
1. a prebuilt wheel from the `mamba-ssm` GitHub release (a download, no compiling);
2. if this image's torch has no prebuilt wheel (Kaggle moved to torch 2.11 in Nov 2026), this session
   first switches to torch 2.10, which has one;
3. a wheel compiled in an earlier session and added as input;
4. compiling from source (slow, last resort; the wheel is then kept in the output).

If all fail, everything still runs on the chunked scan (`scan="auto"` picks the kernel only when it
loads) and the reason is printed. All speed numbers name the backend used.

In [ ]:
# Torch versions with a prebuilt mamba-ssm wheel (release v2.3.2.post1, CUDA 12; checked 2026-11-08).
PREBUILT_TORCH = ('2.6', '2.7', '2.8', '2.9', '2.10')
FALLBACK_TORCH = '2.10.0+cu126'
MAMBA_WHEEL = ('https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/'
               'mamba_ssm-2.3.2.post1+cu12torch{torch}cxx11abiTRUE-cp{py}-cp{py}-linux_x86_64.whl')


def kernel_ok(show_error=False):
    # Same check the trainer uses: loads only the kernel module of mamba-ssm (see afml/models/mamba.py).
    r = subprocess.run([PY, '-c', 'import sys; from afml.models.mamba import HAS_CUDA_SCAN, CUDA_SCAN_ERROR; '
                        'print(CUDA_SCAN_ERROR or ""); sys.exit(0 if HAS_CUDA_SCAN else 1)'],
                       cwd=REPO, capture_output=True, text=True)
    if r.returncode and show_error:
        print('mamba-ssm kernel not available:', '\n'.join((r.stdout + r.stderr).strip().splitlines()[-3:]))
    return r.returncode == 0


def torch_info():
    r = subprocess.run([PY, '-c', 'import sys, torch; print(torch.__version__, torch.version.cuda, '
                        '"%d%d" % sys.version_info[:2])'], capture_output=True, text=True, check=True)
    return r.stdout.split()  # version, CUDA version, python tag (e.g. 313)


FAILED_MARK = WHEELS / 'BUILD_FAILED.txt'
if USE_MAMBA_SSM and GPUS[0] is not None and not kernel_ok():
    version, cuda, py = torch_info()
    minor = '.'.join(version.split('+')[0].split('.')[:2])
    try:  # 1-2) prebuilt wheel, switching torch first if this torch has none
        if minor not in PREBUILT_TORCH or not cuda.startswith('12'):
            print(f'torch {version} (CUDA {cuda}) has no prebuilt mamba-ssm wheel: '
                  f'installing torch {FALLBACK_TORCH} for this session')
            sh(f'pip install -q "torch=={FALLBACK_TORCH}" --index-url https://download.pytorch.org/whl/cu126 '
               f'--extra-index-url https://pypi.org/simple')
            minor = '2.10'
        sh('pip install -q einops')
        sh(f'pip install -q --no-deps "{MAMBA_WHEEL.format(torch=minor, py=py)}"')
    except RuntimeError as e:
        print('prebuilt mamba-ssm wheel failed:', e)
    if not kernel_ok():  # 3) a wheel compiled in an earlier session
        for w in sorted(WHEELS.glob('mamba_ssm-*.whl')):
            sh(f'pip install -q --no-deps {shlex.quote(str(w))}', ok=(0, 1))
    if not kernel_ok() and not FAILED_MARK.exists():  # 4) compile from source
        WHEELS.mkdir(parents=True, exist_ok=True)
        try:
            sh('pip install -q ninja packaging einops')
            sh(f'MAX_JOBS=4 timeout 5400 pip wheel --no-deps --no-build-isolation -w {WHEELS} mamba-ssm')
            sh(f'pip install -q --no-deps {WHEELS}/mamba_ssm-*.whl')
        except RuntimeError as e:
            FAILED_MARK.write_text(f'{time.ctime()}: {e}\n')
            print('mamba-ssm could not be compiled either; Mamba falls back to the slow chunked scan.', e)
    elif not kernel_ok():
        print('mamba-ssm failed to compile in an earlier session (see', FAILED_MARK, '); not retrying.')
HAS_KERNEL = GPUS[0] is not None and kernel_ok(show_error=USE_MAMBA_SSM)
print('torch', torch_info()[0], '| mamba-ssm fused kernel available:', HAS_KERNEL)
sh('python -m pytest -q')   # includes the kernel-vs-reference test when the kernel is installed

## Stage `data`: build the dataset (once)

In [ ]:
if 'data' in STAGES:
    if not (DATA / 'meta.json').exists():
        sh(f'{PY} -m afml.data.build --config configs/data.json --out {DATA}')
    elif not (DATA / 'pg19_test_offsets.npy').exists():
        sh(f'{PY} -m afml.data.build --config configs/data.json --out {DATA} --pg19-only')
if (DATA / 'meta.json').exists():
    meta = json.loads((DATA / 'meta.json').read_text())
    print(DATA, json.dumps(meta['splits'], indent=2))
elif STAGES & {'smoke', 'sweep', 'S', 'M', 'eval'}:
    raise RuntimeError('No dataset found. Run the data stage first, then add that version\'s output '
                       'with "Add Input" in every later session.')

## Stage `smoke`: 20-step GPU check before spending GPU hours

Trains `S_transformer` and `S_mamba` for 20 steps (Mamba once per scan backend: `chunked`, and `cuda` if
`mamba-ssm` is installed). Reports steady-state tokens/sec, the micro-batch the trainer settled on, and
peak GPU memory (`torch.cuda.max_memory_allocated()`), then estimates hours for every full run and stage.
Two settings are smaller than a real run only to keep this short: 8 gradient-accumulation steps instead
of 32 (tokens/sec does not depend on it) and 0.5M evaluation tokens instead of 2M (eval time is
measured per token and scaled back up). The slow chunked-scan Mamba is measured over 8 steps of one
micro-batch each. Smoke runs are named `smoke_*`, so `collect_results.py` ignores them.

In [ ]:
SMOKE_SETS = ['train.eval_every=10', 'train.log_every=5', 'train.grad_accum=8', 'train.eval_tokens=524288']
# The chunked PyTorch scan is much slower on a GPU, so it is measured with fewer and smaller steps.
CHUNKED_SETS = ['train.grad_accum=1', 'train.log_every=2', 'train.eval_every=4', 'train.eval_tokens=65536']
if 'smoke' in STAGES:
    variants = [('S_transformer', 'smoke_S_transformer', [], 20)]
    if HAS_KERNEL:
        variants.append(('S_mamba', 'smoke_S_mamba_cuda', ['model.scan=cuda'], 20))
    variants.append(('S_mamba', 'smoke_S_mamba_chunked', ['model.scan=chunked'] + CHUNKED_SETS, 8))
    jobs = []
    for cfg, name, extra, steps in variants:
        out = RUNS / name
        shutil.rmtree(out, ignore_errors=True)  # always a fresh measurement on this session's GPU
        j = train_job(name, f'configs/{cfg}.json', out, SMOKE_SETS + extra, keep=())
        j['cmd'] += ['--max-steps', str(steps)]
        j['filter'] = False
        jobs.append(j)
    run_jobs(jobs)
    smoke_runs = ' '.join(str(RUNS / name) for _, name, _, _ in variants)
    sh(f'{PY} scripts/smoke_report.py --runs {smoke_runs} --gpus {len(GPUS)} '
       f'--session-hours {SESSION_HOURS} --out {RESULTS}/smoke')

## Stage `sweep`: learning-rate sweep at scale S (47M)

Transformer x 5 and Mamba x 4 learning rates, 30M tokens each (same batch as the main runs), chosen on
**validation** loss. Each sweep run warms up for the same number of steps as the main run (51), not 2% of
its own 228 steps. A run whose loss blows up is printed as `diverged` (that learning rate is too high):
normal in a sweep, recorded in the CSV, and skipped when picking the best. Writes
`results/S/lr_sweep.csv` and `results/S/best_lr.json`. Learning rates within 0.01 validation loss of the
best count as tied and the lowest is used (the full run is 11x longer). A warning is printed if the chosen
LR sits at the edge of the range. Mamba-LTI uses Mamba's learning rate.

In [ ]:
from sweep_lr import DEFAULT_LRS, DEFAULT_TOKENS, best_from_csv, sweep_jobs


def best_lr():
    # Recomputed from the sweep's CSV with the rule in scripts/sweep_lr.py (tied LRs -> the lowest), so a
    # rule change never needs the sweep to be rerun. Also rewrites results/S/best_lr.json to match.
    f = RESULTS / 'S' / 'lr_sweep.csv'
    if not f.exists():
        raise RuntimeError('results/S/lr_sweep.csv not found: finish the sweep stage first, '
                           'or add the output of the session that finished it as input.')
    best = best_from_csv(f)
    (RESULTS / 'S' / 'best_lr.json').write_text(json.dumps(best, indent=2))
    return best


if 'sweep' in STAGES:
    jobs = []
    for arch, lr, out, cmd in sweep_jobs('S', ['transformer', 'mamba'], DEFAULT_LRS, DEFAULT_TOKENS, DATA, RUNS):
        jobs.append({'name': out.name, 'cmd': cmd, 'out': out, 'log': out / 'train.log',
                     'pausable': True, 'filter': True, 'keep': (), 'diverged_ok': True})
    run_jobs(jobs)
    missing = [j['name'] for j in jobs if not finished(j['out'])]
    if not missing:
        sh(f'{PY} scripts/sweep_lr.py --scale S --archs transformer mamba --data {DATA} --runs {RUNS} '
           f'--results {RESULTS}')
        print(best_lr())
    else:
        print('Sweep not finished yet (missing: ' + ', '.join(missing) + '). Run the sweep stage again '
              'next session, with the sweep outputs of every account added as input.')

## Stage `S`: main runs at scale S (~47M), 3 seeds each

In [ ]:
def main_job(scale, arch, seed, lr):
    name = f'{scale}_{arch}_s{seed}'
    return train_job(name, f'configs/{scale}_{arch}.json', RUNS / name, [f'train.seed={seed}', f'train.lr={lr}'])


if 'S' in STAGES:
    lrs = best_lr()
    print('learning rates (from the sweep):', lrs, flush=True)
    run_jobs([main_job('S', arch, seed, lrs[arch])
              for seed in (0, 1, 2) for arch in ('transformer', 'mamba', 'mamba_lti')])

## Stage `M`: scale M (~98M), 1 seed each

The learning-rate sweep runs at S (47M). Larger models need smaller learning rates, so M uses the S
optimum x 0.6, roughly the ratio of the default LRs in `scripts/make_configs.py` (4e-4 / 6e-4).
If there is spare GPU time, sweep M too with `scripts/sweep_lr.py --scale M`.

In [ ]:
if 'M' in STAGES:
    lrs = best_lr()
    run_jobs([main_job('M', arch, 0, round(lrs[arch] * 0.6, 6)) for arch in ('transformer', 'mamba')])

## Stage `synthetic`: MQAR and selective copying (small models, both GPUs)

In [ ]:
SYN = RESULTS / 'synthetic'
if 'synthetic' in STAGES:
    tasks = {'selective_copy': '--d-models 64 --seq-lens 64 128 256 512',
             'mqar': '--d-models 64 128 256 --seq-lens 64 128 256 512'}
    run_jobs([{'name': f'synthetic_{t}', 'out': None, 'log': SYN / f'{t}.log', 'pausable': False,
               'filter': False, 'keep': (),
               'cmd': [PY, '-m', 'afml.eval.synthetic', '--task', t, '--archs', 'transformer', 'mamba',
                       'mamba_lti', *args.split(), '--out', str(SYN / f'{t}.csv')]}
              for t, args in tasks.items()], min_hours=1.0)

## Stage `eval`: test perplexity, PG-19 long context, efficiency, delta analysis

The test split is scored **once** per run: a scale is skipped when its CSV already covers every finished
run. The efficiency benchmark uses untrained weights (timing only) and the GPU to itself; it runs once
(delete `results/bench/bench.csv` to redo it, e.g. after `mamba-ssm` becomes available).

In [ ]:
def finished(pattern):
    return sorted(str(p.parent) for p in RUNS.glob(f'{pattern}/summary.json'))


def covered(csv_path, runs):
    if not Path(csv_path).exists():
        return False
    with open(csv_path) as f:
        done = {r['run'] for r in csv.DictReader(f)}
    return {Path(r).name for r in runs} <= done


if 'eval' in STAGES:
    if hours_left() < 1.5:
        print(f'WARNING: only {hours_left():.1f} h left; evaluation steps cannot pause.')
    for scale in ('S', 'M'):
        runs = finished(f'{scale}_*_s*')
        if not runs:
            continue
        out = RESULTS / scale
        if not covered(out / 'perplexity.csv', runs):
            sh(f'{PY} -m afml.eval.perplexity --runs {" ".join(runs)} --data {DATA} --split test '
               f'--out {out}/perplexity.csv')
        seed0 = finished(f'{scale}_*_s0')
        if seed0 and not covered(out / 'long_context.csv', seed0):
            sh(f'{PY} -m afml.eval.long_context --runs {" ".join(seed0)} --data {DATA} --max-len 16384 '
               f'--out {out}/long_context.csv')
    if (RESULTS / 'bench' / 'bench.csv').exists():
        print('efficiency benchmark done earlier:', RESULTS / 'bench' / 'bench.csv')
    else:
        sh(f'{PY} -m afml.eval.bench --configs configs/M_transformer.json configs/M_mamba.json '
           f'--lengths 512 1024 2048 4096 8192 16384 32768 --batches 1 8 --out {RESULTS}/bench/bench.csv')
    for scale in ('M', 'S'):  # delta / attention analysis on the largest finished pair
        mam, tf = RUNS / f'{scale}_mamba_s0', RUNS / f'{scale}_transformer_s0'
        if (mam / 'summary.json').exists() and (tf / 'summary.json').exists():
            sh(f'{PY} -m afml.eval.analysis --mamba {mam} --transformer {tf} --data {DATA} '
               f'--out {RESULTS}/analysis/{scale}')
            break

## Stage `figures`, and the session summary (always runs)

In [ ]:
sh(f'{PY} scripts/collect_results.py --runs {RUNS} --results {RESULTS} --data {DATA}')
if 'figures' in STAGES:
    sh(f'{PY} scripts/make_figures.py --results {RESULTS} --out {RESULTS}/figures')
archive = shutil.make_archive(str(BASE / 'results'), 'zip', RESULTS)

print('\nRuns in this output:')
for cfg in sorted(RUNS.rglob('config.json')):
    run = cfg.parent
    if (run / 'summary.json').exists():
        s = json.loads((run / 'summary.json').read_text())
        print(f'  done    {run.relative_to(RUNS)}: best val loss {s["best_val_loss"]:.4f}, '
              f'{s["train_seconds"] / 3600:.1f} h, scan {s.get("scan_backend") or "-"}')
    elif (run / 'diverged.json').exists():
        d = json.loads((run / 'diverged.json').read_text())
        print(f'  diverged {run.relative_to(RUNS)}: loss became NaN/inf at step {d["step"]}')
    else:
        rows = list(csv.DictReader(open(run / 'metrics.csv'))) if (run / 'metrics.csv').exists() else []
        print(f'  paused  {run.relative_to(RUNS)}: at step {rows[-1]["step"] if rows else 0} (resumes next session)')
print(f'\nSession used {(time.time() - T0) / 3600:.1f} h. Results archive: {archive}')
sh(f'du -sh {BASE}', cwd='/')
print('Next session: Add Input -> this version\'s output AND the data version\'s output.')

from IPython.display import Image, Markdown, display
if (RESULTS / 'smoke' / 'smoke_report.md').exists():
    display(Markdown((RESULTS / 'smoke' / 'smoke_report.md').read_text()))
for f in sorted((RESULTS / 'figures').glob('*.png')):
    display(Image(filename=str(f)))